In [22]:
from __future__ import annotations
import numpy as np
from pathlib import Path
import pandas as pd
import h5py
import BBH_SIM
import time

In [28]:
!cd /home/synodii/BBH_merger_sim
!python run.py

[1/9] merged=False
Traceback (most recent call last):
  File "/home/synodii/BBH_merger_sim/run.py", line 7, in <module>
    run_all(
  File "/home/synodii/BBH_merger_sim/BBH_SIM/runsimulation.py", line 66, in run_all
    sim.save_results(store, run_id=run_id)
  File "/home/synodii/BBH_merger_sim/BBH_SIM/simulation.py", line 161, in save_results
    return store.append(
  File "/home/synodii/BBH_merger_sim/BBH_SIM/datastorage.py", line 63, in append
    ds = f[self.DATASET]
  File "h5py/_objects.pyx", line 54, in h5py._objects.with_phil.wrapper
  File "h5py/_objects.pyx", line 55, in h5py._objects.with_phil.wrapper
  File "/home/synodii/anaconda3/lib/python3.9/site-packages/h5py/_hl/group.py", line 305, in __getitem__
    oid = h5o.open(self.id, self._e(name), lapl=self._lapl)
  File "h5py/_objects.pyx", line 54, in h5py._objects.with_phil.wrapper
  File "h5py/_objects.pyx", line 55, in h5py._objects.with_phil.wrapper
  File "h5py/h5o.pyx", line 190, in h5py.h5o.open
KeyError: "Unable t

In [29]:
df = pd.read_hdf("BBH_SIM/Results/bbh_results.h5", key="results")
print(df)
print(df.dtypes)

KeyError: 'No object named results in the file'

In [30]:
df = pd.read_hdf("BBH_SIM/Results/bbh_results_int1.h5", key="results")
print(df)
print(df.dtypes)

   run_id  bh1_mass_msol  bh2_mass_msol  impact_parameter_au  \
0       1       100000.0       100000.0                  0.0   
1       1       100000.0       100000.0                  0.0   

   bh1_schwarzschild_radius_km  bh2_schwarzschild_radius_km  \
0                 1.485232e-25                 1.485232e-25   
1                 1.485232e-25                 1.485232e-25   

   simulation_duration_yr  bh1_v_init_x_kms  bh1_v_init_y_kms  \
0             1000.000317        14989.6229               0.0   
1             1000.000317        14989.6229               0.0   

   bh1_v_init_total_kms  ...  bh2_v_final_x_kms  bh2_v_final_y_kms  \
0            14989.6229  ...      -6.260931e-28                0.0   
1            14989.6229  ...      -6.260931e-28                0.0   

   bh2_v_final_total_kms  bh2_v_final_unit_x  bh2_v_final_unit_y  \
0           6.260931e-28                -1.0                 0.0   
1           6.260931e-28                -1.0                 0.0   

   bh

In [33]:
"""
generate_table.py
Run from /home/synodii/BBH_merger_sim:
    python generate_table.py

Reads BBH_SIM/Results/bbh_results.h5 and writes a publication-ready
HTML table to BBH_SIM/Results/results_table.html
"""

import sys

# ── Load data ─────────────────────────────────────────────────────────────────
H5_PATH  = Path("BBH_SIM/Results/bbh_results_int1.h5")
OUT_PATH = Path("BBH_SIM/Results/results_table.html")

try:
    from BBH_SIM.datastorage import data_storage
    store = data_storage(H5_PATH)
    df = store.to_dataframe()
except Exception as e:
    print(f"Error loading HDF5: {e}")
    sys.exit(1)

if len(df) == 0:
    print("No rows found in HDF5 file.")
    sys.exit(1)

# ── Select and rename columns for the paper table ────────────────────────────
# Adjust which columns to show / hide here
DISPLAY_COLS = {
    "run_id":                       "Run ID",
    "bh1_mass_msol":                "M₁ (M☉)",
    "bh2_mass_msol":                "M₂ (M☉)",
    "impact_parameter_au":          "b (AU)",
    "bh1_schwarzschild_radius_km":  "R_s1 (km)",
    "bh2_schwarzschild_radius_km":  "R_s2 (km)",
    "bh1_v_init_total_kms":         "v₁ᵢ (km/s)",
    "bh2_v_init_total_kms":         "v₂ᵢ (km/s)",
    "bh1_v_final_total_kms":        "v₁f (km/s)",
    "bh2_v_final_total_kms":        "v₂f (km/s)",
    "bh1_deflection_angle_deg":     "θ₁ (°)",
    "bh2_deflection_angle_deg":     "θ₂ (°)",
    "merged":                       "Merged",
    "nearest_approach_dist_au":     "d_min (AU)",
    "nearest_approach_time_s":      "t_min (s)",
    "remaining_dist_for_merger_au": "Δd_merger (AU)",
    "simulation_duration_yr":       "t_sim (yr)",
}

# Keep only columns that exist in the dataframe
available = {k: v for k, v in DISPLAY_COLS.items() if k in df.columns}
table_df  = df[list(available.keys())].copy()
table_df.columns = list(available.values())

# ── Format numbers ────────────────────────────────────────────────────────────
def fmt(val, col):
    if col == "Run ID":
        return str(int(val))
    if col == "Merged":
        return "Yes" if int(val) == 1 else "No"
    if isinstance(val, float) and np.isnan(val):
        return "—"
    if abs(val) >= 1e4 or (abs(val) < 1e-3 and val != 0):
        return f"{val:.3e}"
    return f"{val:.4f}"

# Build formatted rows
rows_html = ""
for i, row in table_df.iterrows():
    bg = '#0a0f1a' if i % 2 == 0 else '#0d1520'
    cells = ""
    for col, val in zip(table_df.columns, row):
        merged_style = ""
        if col == "Merged":
            color = "#4ade80" if val == 1 else "#f87171"
            merged_style = f'style="color:{color};font-weight:600;"'
        cells += f'<td {merged_style}>{fmt(val, col)}</td>'
    rows_html += f'<tr style="background:{bg}">{cells}</tr>\n'

# Header
headers_html = "".join(f"<th>{c}</th>" for c in table_df.columns)

# ── Caption / summary stats ───────────────────────────────────────────────────
n_total   = len(df)
n_merged  = int(df["merged"].sum())
n_flyby   = n_total - n_merged

# ── Build HTML ────────────────────────────────────────────────────────────────
html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>BBH Simulation Results</title>
<link href="https://fonts.googleapis.com/css2?family=EB+Garamond:ital,wght@0,400;0,600;1,400&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<style>
  :root {{
    --bg:        #070c14;
    --surface:   #0a0f1a;
    --border:    #1e2d45;
    --accent:    #3b82f6;
    --accent2:   #93c5fd;
    --text:      #e2e8f0;
    --subtext:   #94a3b8;
    --head-bg:   #111827;
  }}

  * {{ box-sizing: border-box; margin: 0; padding: 0; }}

  body {{
    background: var(--bg);
    color: var(--text);
    font-family: 'EB Garamond', Georgia, serif;
    font-size: 15px;
    padding: 48px 32px;
    min-height: 100vh;
  }}

  .wrapper {{
    max-width: 1400px;
    margin: 0 auto;
  }}

  /* ── Header ── */
  .doc-header {{
    border-bottom: 1px solid var(--border);
    padding-bottom: 24px;
    margin-bottom: 32px;
  }}

  .doc-header h1 {{
    font-size: 22px;
    font-weight: 600;
    letter-spacing: 0.02em;
    color: var(--text);
    margin-bottom: 6px;
  }}

  .doc-header .subtitle {{
    font-style: italic;
    color: var(--subtext);
    font-size: 14px;
  }}

  /* ── Summary stats ── */
  .stats {{
    display: flex;
    gap: 24px;
    margin-bottom: 28px;
    flex-wrap: wrap;
  }}

  .stat-card {{
    background: var(--surface);
    border: 1px solid var(--border);
    border-radius: 6px;
    padding: 14px 22px;
    min-width: 140px;
  }}

  .stat-card .label {{
    font-size: 11px;
    text-transform: uppercase;
    letter-spacing: 0.1em;
    color: var(--subtext);
    margin-bottom: 4px;
    font-family: 'JetBrains Mono', monospace;
  }}

  .stat-card .value {{
    font-size: 22px;
    font-weight: 600;
    color: var(--accent2);
  }}

  /* ── Table container ── */
  .table-wrap {{
    overflow-x: auto;
    border: 1px solid var(--border);
    border-radius: 8px;
  }}

  table {{
    width: 100%;
    border-collapse: collapse;
    font-family: 'JetBrains Mono', monospace;
    font-size: 12.5px;
  }}

  thead tr {{
    background: var(--head-bg);
  }}

  thead th {{
    padding: 12px 14px;
    text-align: right;
    font-weight: 500;
    font-size: 11px;
    text-transform: uppercase;
    letter-spacing: 0.06em;
    color: var(--accent2);
    border-bottom: 1px solid var(--border);
    white-space: nowrap;
  }}

  thead th:first-child {{
    text-align: left;
  }}

  tbody td {{
    padding: 9px 14px;
    text-align: right;
    color: var(--text);
    border-bottom: 1px solid var(--border);
    white-space: nowrap;
  }}

  tbody td:first-child {{
    text-align: left;
    color: var(--subtext);
  }}

  tbody tr:last-child td {{
    border-bottom: none;
  }}

  tbody tr:hover td {{
    background: #131e30 !important;
  }}

  /* ── Caption ── */
  .caption {{
    margin-top: 16px;
    font-size: 13px;
    font-style: italic;
    color: var(--subtext);
    line-height: 1.6;
    max-width: 900px;
  }}

  .caption strong {{
    font-style: normal;
    color: var(--text);
  }}

  /* ── Print styles ── */
  @media print {{
    body {{
      background: white;
      color: black;
      padding: 20px;
    }}
    .stats {{ display: none; }}
    table {{
      font-size: 9pt;
      color: black;
    }}
    thead th {{
      background: #f0f0f0;
      color: black;
      border: 1px solid #999;
    }}
    tbody td {{
      border: 1px solid #ccc;
      color: black;
      background: white !important;
    }}
    tbody tr:nth-child(even) td {{
      background: #f9f9f9 !important;
    }}
    .caption {{
      color: #333;
    }}
  }}
</style>
</head>
<body>
<div class="wrapper">

  <div class="doc-header">
    <h1>Binary Black Hole Merger Simulation Results</h1>
    <div class="subtitle">
      Newtonian gravity &nbsp;·&nbsp; 2D &nbsp;·&nbsp;
      T<sub>end</sub> = 1000 yr &nbsp;·&nbsp; dt = 10<sup>4</sup> s
    </div>
  </div>

  <div class="stats">
    <div class="stat-card">
      <div class="label">Total Runs</div>
      <div class="value">{n_total}</div>
    </div>
    <div class="stat-card">
      <div class="label">Mergers</div>
      <div class="value" style="color:#4ade80">{n_merged}</div>
    </div>
    <div class="stat-card">
      <div class="label">Fly-bys</div>
      <div class="value" style="color:#f87171">{n_flyby}</div>
    </div>
  </div>

  <div class="table-wrap">
    <table>
      <thead>
        <tr>{headers_html}</tr>
      </thead>
      <tbody>
        {rows_html}
      </tbody>
    </table>
  </div>

  <p class="caption">
    <strong>Table 1.</strong>
    Summary of binary black hole (BBH) flyby/merger simulations.
    Columns: Run ID; BH masses M₁, M₂ in solar masses (M☉);
    impact parameter <em>b</em> (AU); Schwarzschild radii R_s1, R_s2 (km);
    initial and final speeds v₁ᵢ, v₂ᵢ, v₁f, v₂f (km s⁻¹);
    deflection angles θ₁, θ₂ (degrees);
    merger outcome; closest approach distance d_min (AU) and time t_min (s);
    remaining distance to merger threshold Δd_merger (AU);
    total simulated duration t_sim (yr).
    Velocities converted from SI (m s⁻¹) for display.
  </p>

</div>
</body>
</html>
"""

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUT_PATH.write_text(html, encoding="utf-8")
print(f"Table written to {OUT_PATH.resolve()}")
print(f"Rows: {n_total}  |  Mergers: {n_merged}  |  Fly-bys: {n_flyby}")

Table written to /home/synodii/BBH_merger_sim/BBH_SIM/Results/results_table.html
Rows: 2  |  Mergers: 0  |  Fly-bys: 2
